Nama    : Ferdy Ardiansyah

NIM     : 231011402416

UTS     : Teknik Kompilasi

JAWABAN TUGAS 1

menambahkan karakter ^ ke dalam regular expression agar compiler mengenali simbol pangkat sebagai token yang valid.

In [ ]:
import re

class MiniCompiler:
    def __init__(self, source, env):
        # TUGAS 1: Menambahkan '^' ke dalam regex agar dikenali sebagai token
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]', source) + ['?'])
        self._current = None
        self._env = env 
        self._temp_count = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def factor(self):
        token = self._current
        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))
        elif token and token.isalpha():
            if token not in self._env:
                raise ParserError(f"Semantic Error: Undefined variable '{token}'")
            self.advance()
            return Var(token)
        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node
        raise ParserError(f"Unexpected token: {token}")

TUGAS 2 

Implementasi Fungsi power()

Fungsi ini bertugas menangani logika operasi pangkat. Kita menggunakan perulangan while agar bisa menangani rantaian pangkat jika ada.

In [ ]:
    def power(self):
        node = self.factor()
        while self._current == '^':
            op = self._current
            self.advance()
            # Pangkat bersifat right-associative atau tetap memanggil power() 
            node = BinOp(left=node, op=op, right=self.factor())
        return node

    def term(self):

TUGAS 3: Menghubungkan Hierarki ke term()

Agar urutan matematika benar (Pangkat dikerjakan sebelum Perkalian), 

kita harus mengubah fungsi term() agar memanggil power() terlebih dahulu, 

bukan langsung ke factor().

In [ ]:
        # Sekarang term memanggil power()
        node = self.power() 
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.power()) #ini juga memanggil power()
        return node

    def expr(self):
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    def generate_tac(self, node):
        if isinstance(node, Num): return str(node.value)
        if isinstance(node, Var): return node.name
        
        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)
        
        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val}")
        return temp_name

Jawaban Pertanyaan Refleksi

1.Mengapa fungsi power() harus dipanggil di dalam term(), bukan sebaliknya? Jelaskan kaitannya dengan Operator Precedence.

Fungsi power() harus dipanggil di dalam fungsi term() karena dalam teknik kompilasi menggunakan Top-Down Parsing, fungsi yang dipanggil lebih dalam atau lebih akhir dalam hirarki secara otomatis memiliki prioritas (precedence) yang lebih tinggi. Dalam struktur ini, expr memanggil term untuk menangani penjumlahan, kemudian term memanggil power untuk menangani perkalian, dan akhirnya power memanggil factor untuk menangani pangkat. Karena term memanggil power sebelum melakukan operasinya sendiri, maka operasi pangkat akan diproses dan "diikat" lebih kuat dalam Abstract Syntax Tree (AST) dibandingkan perkalian atau pembagian.

2.Apa yang terjadi pada fase Analisis Semantik jika variabel z digunakan dalam kode sumber tetapi tidak ada di symbol_table?

Apabila terdapat variabel seperti z yang digunakan dalam kode sumber namun tidak terdaftar di dalam symbol_table, maka pada fase analisis semantik program akan melemparkan kesalahan atau Exception berupa ParserError dengan pesan "Semantic Error: Undefined variable 'z'". Hal ini terjadi karena compiler melakukan pemeriksaan validitas identitas variabel pada tahap analisis semantik—yang dalam implementasi ini digabung saat proses parsing faktor—untuk memastikan bahwa setiap variabel yang dipanggil sudah dideklarasikan atau didefinisikan di dalam lingkungan (environment) yang tersedia.

3. Jelaskan mengapa dalam TAC, instruksi untuk a ^ 2 harus muncul sebelum instruksi untuk +.
Dalam format Three Address Code (TAC), instruksi untuk operasi pangkat seperti a ^ 2 harus muncul sebelum instruksi untuk penjumlahan + karena TAC merepresentasikan urutan eksekusi kode secara linear. Mengingat operator pangkat memiliki prioritas yang lebih tinggi daripada penjumlahan, maka hasil dari a ^ 2 harus dihitung terlebih dahulu dan disimpan ke dalam variabel sementara, misalnya t1. Hasil dari variabel t1 inilah yang kemudian digunakan sebagai input untuk operasi penjumlahan berikutnya; jika operasi penjumlahan dikerjakan lebih dulu, maka hasil matematisnya akan menyimpang dari aturan urutan operasi matematika yang baku (PEMDAS/BODMAS).